In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device


'cuda'

In [3]:
train_df = pd.read_csv("../data/raw/train.csv")
X_img = np.load("../data/processed/train_image_embeddings.npy")

y = np.log1p(train_df["price"].values)


In [4]:
tabular_features = [
    "bedrooms","bathrooms","sqft_living","sqft_lot","floors",
    "waterfront","view","condition","grade",
    "sqft_above","sqft_basement","yr_built","yr_renovated",
    "lat","long"
]

X_tab = train_df[tabular_features].values


In [5]:
scaler = StandardScaler()
X_tab = scaler.fit_transform(X_tab)


In [6]:
X_tab_tr, X_tab_val, X_img_tr, X_img_val, y_tr, y_val = train_test_split(
    X_tab, X_img, y, test_size=0.2, random_state=42
)


In [7]:
X_tab_tr = torch.tensor(X_tab_tr, dtype=torch.float32)
X_tab_val = torch.tensor(X_tab_val, dtype=torch.float32)
X_img_tr = torch.tensor(X_img_tr, dtype=torch.float32)
X_img_val = torch.tensor(X_img_val, dtype=torch.float32)
y_tr = torch.tensor(y_tr, dtype=torch.float32).view(-1, 1)
y_val = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)


In [8]:
class MultiModalDataset(Dataset):
    def __init__(self, X_tab, X_img, y):
        self.X_tab = X_tab
        self.X_img = X_img
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_tab[idx], self.X_img[idx], self.y[idx]


In [9]:
train_ds = MultiModalDataset(X_tab_tr, X_img_tr, y_tr)
val_ds = MultiModalDataset(X_tab_val, X_img_val, y_val)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)


In [10]:
class MultiModalRegressor(nn.Module):
    def __init__(self):
        super().__init__()

        self.tab_net = nn.Sequential(
            nn.Linear(15, 64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )

        self.img_net = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 64)
        )

        self.fusion = nn.Sequential(
            nn.Linear(32 + 64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x_tab, x_img):
        tab_feat = self.tab_net(x_tab)
        img_feat = self.img_net(x_img)
        fused = torch.cat([tab_feat, img_feat], dim=1)
        return self.fusion(fused)


In [11]:
model = MultiModalRegressor().to(device)


In [12]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [13]:
epochs = 25

for epoch in range(epochs):
    model.train()
    train_loss = 0

    for x_tab, x_img, y in train_loader:
        x_tab, x_img, y = x_tab.to(device), x_img.to(device), y.to(device)

        optimizer.zero_grad()
        preds = model(x_tab, x_img)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x_tab, x_img, y in val_loader:
            x_tab, x_img, y = x_tab.to(device), x_img.to(device), y.to(device)
            preds = model(x_tab, x_img)
            val_loss += criterion(preds, y).item()

    val_loss /= len(val_loader)

    print(f"Epoch {epoch+1:02d} | Train MSE: {train_loss:.4f} | Val MSE: {val_loss:.4f}")


Epoch 01 | Train MSE: 9.2590 | Val MSE: 0.9635
Epoch 02 | Train MSE: 0.6879 | Val MSE: 0.5208
Epoch 03 | Train MSE: 0.4200 | Val MSE: 0.3786
Epoch 04 | Train MSE: 0.3229 | Val MSE: 0.3333
Epoch 05 | Train MSE: 0.2612 | Val MSE: 0.2562
Epoch 06 | Train MSE: 0.2170 | Val MSE: 0.2129
Epoch 07 | Train MSE: 0.1729 | Val MSE: 0.1667
Epoch 08 | Train MSE: 0.1416 | Val MSE: 0.1448
Epoch 09 | Train MSE: 0.1108 | Val MSE: 0.1440
Epoch 10 | Train MSE: 0.0927 | Val MSE: 0.1080
Epoch 11 | Train MSE: 0.0829 | Val MSE: 0.0890
Epoch 12 | Train MSE: 0.0668 | Val MSE: 0.0920
Epoch 13 | Train MSE: 0.0603 | Val MSE: 0.0781
Epoch 14 | Train MSE: 0.0548 | Val MSE: 0.0706
Epoch 15 | Train MSE: 0.0529 | Val MSE: 0.0676
Epoch 16 | Train MSE: 0.0511 | Val MSE: 0.0627
Epoch 17 | Train MSE: 0.0476 | Val MSE: 0.0634
Epoch 18 | Train MSE: 0.0486 | Val MSE: 0.0658
Epoch 19 | Train MSE: 0.0480 | Val MSE: 0.0620
Epoch 20 | Train MSE: 0.0465 | Val MSE: 0.0577
Epoch 21 | Train MSE: 0.0426 | Val MSE: 0.0582
Epoch 22 | Tr

In [14]:
model.eval()
with torch.no_grad():
    preds = model(X_tab_val.to(device), X_img_val.to(device)).cpu().numpy()

rmse_nn = np.sqrt(np.mean((preds - y_val.numpy())**2))
from sklearn.metrics import r2_score
r2_nn = r2_score(y_val.numpy(), preds)

rmse_nn, r2_nn


(np.float32(0.2510659), 0.7715772986412048)